# Push to Chinook to Hugging face

We'll convert the tables in `chinook` to parquet files in a directory and then push the directory to Hugging Face.


## Chinook data set

See the lecture on SQLite3 using the Chinook data set to set up the software, database, and tables, as well as for the links to ancillary information about the data set.


In [ ]:
import sqlite3 as db
import pandas as pd
import contextlib
import requests
from google.colab import userdata
import os


In [ ]:
%%bash
curl -s -O https://www.sqlitetutorial.net/wp-content/uploads/2018/03/chinook.zip


In [ ]:
!unzip -u chinook.zip


In [ ]:
!ls -l


Verify sqlite works.

In [ ]:
query = '''
  SELECT
    name
  FROM
    sqlite_master
  WHERE
    type='table' AND
    name not like "sqlite_%"
;
'''
print(query)

In [ ]:
with contextlib.closing(db.connect("chinook.db")) as db_con:
  tables = pd.read_sql_query( query , db_con)

tables


## Read each table and convert to parquet


In [ ]:
# create target folder
!mkdir -p chinook


In [ ]:
with contextlib.closing(db.connect("chinook.db")) as db_con:
  for table in tables["name"]:
    print(table)
    query = f'''
      select * from {table} ;
    '''
    df = pd.read_sql_query( query, db_con)
    display(df.shape)
    output_path = f'./chinook/{table}.parquet'
    df.to_parquet(output_path, index=False)


In [ ]:
# verify
!ls -l chinook


## Read from parquet



In [ ]:
files = !ls -1 chinook/*.parquet
files


In [ ]:
for table in files:
  print(table)
  df = pd.read_parquet( table )
  display(df.shape)



## Authenticate to Hugging Face


In [ ]:
# Retrieve the token from Colab's secret manager
os.environ["HF_TOKEN"] = userdata.get('hf_token')
_ = os.environ["HF_TOKEN"]
f"{_[:5]} ... {_[-3:]}"

'hf_zp ... QPb'

In [ ]:
# Log in automatically without an interactive prompt
!hf auth login --token $HF_TOKEN


Hint: A new version of huggingface_hub (1.22.0) is available! You are using version 1.20.1.
To update, run: hf update
The token has not been saved to the git credentials helper. Pass `add_to_git_credential=True` in this function directly or `--add-to-git-credential` if using via `hf`CLI if you want to set the git credential as well.
Token is valid (permission: fineGrained).
The token `Colab` has been saved to /root/.cache/huggingface/stored_tokens
Your token has been saved to /root/.cache/huggingface/token
Login successful.
Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


In [ ]:
# set up account name
hf_account = os.environ["hf_account"] = userdata.get('hf_account')
hf_account


'rwcitek'

In [ ]:
# set up repo name
hf_repo = os.environ["hf_repo"] = "Data_Science-21"
hf_repo


'Data_Science-21'

In [ ]:
# set up file
hf_parquet = os.environ["hf_parquet"] = 'chinook'
hf_parquet


'chinook'

## Push to Hugging Face


In [ ]:
%%capture hf_upload
%%bash

hf upload \
  --type dataset \
  ${hf_account}/${hf_repo} \
  ./${hf_parquet}/ \
  ${hf_parquet}/


In [ ]:
print(hf_upload.stdout)


✓ Uploaded
  url: https://huggingface.co/datasets/rwcitek/Data_Science-21/commit/35885a17f644cade5a38c48991c46385570ad719



## Read parquet from Hugging Face


In [ ]:
hf_url = 'https://huggingface.co/datasets/rwcitek/Data_Science-21/resolve/main/chinook/employees.parquet?download=true'
hf_url


'https://huggingface.co/datasets/rwcitek/Data_Science-21/resolve/main/chinook/employees.parquet?download=true'

In [ ]:
df3 = pd.read_parquet( hf_url )
df3.shape


(8, 15)

In [ ]:
df3.iloc[:,:5].info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8 entries, 0 to 7
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   EmployeeId  8 non-null      int64  
 1   LastName    8 non-null      object 
 2   FirstName   8 non-null      object 
 3   Title       8 non-null      object 
 4   ReportsTo   7 non-null      float64
dtypes: float64(1), int64(1), object(3)
memory usage: 452.0+ bytes
